In [2]:
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pickle

In [3]:
#import dataset
df = pd.read_csv("Churn_Modelling.csv")

In [4]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [5]:
df = df.drop(["RowNumber","CustomerId","Surname"],axis=1)

In [6]:
label_encoding_gender = LabelEncoder()
df["Gender"] = label_encoding_gender.fit_transform(df["Gender"])

In [7]:
#One hot encoding 'Geography'
from sklearn.preprocessing import OneHotEncoder


Onehot_encoding_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = Onehot_encoding_geo.fit_transform(df[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=Onehot_encoding_geo.get_feature_names_out(["Geography"]))

In [8]:
df = pd.concat([df.drop("Geography",axis=1),geo_encoded_df],axis=1)

In [9]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [10]:
X = df.drop('EstimatedSalary',axis=1)
Y = df["EstimatedSalary"]

In [11]:
## Split dataset into traing and testing set

X_train,X_test,y_train,y_test  = train_test_split(X,Y,test_size=0.2,random_state=42)

In [12]:
scaler = StandardScaler()
X_train_scaler = scaler.fit_transform(X_train)
X_test_scaler = scaler.transform((X_test))

In [13]:
with open('lable_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoding_gender,file)

with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(Onehot_encoding_geo,file)

with open('scaler.pkl',"wb") as file:
    pickle.dump(scaler,file)

# ANN Regression Problem Statment 

In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from tensorflow.keras.layers import Dense
import datetime

In [15]:
# build your ANN Model
model = Sequential([
    Dense(64, activation='relu',input_shape=(X_train.shape[1],)), ##HL1
    Dense(32, activation='relu'), ## HL2
    Dense(1)  ## Output Layer
])

In [16]:
## complie the model
model.compile(optimizer="adam",loss="mean_absolute_error",metrics=["mae"])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [17]:
log_dir = "regrassionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [18]:
Early_stoping_callback=EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [19]:
# train the model
history = model.fit(
    X_train,y_train,
    validation_data=(X_test,y_test),
    epochs=100,
    callbacks = [tensorflow_callback,Early_stoping_callback]
)

Epoch 1/100


250/250 [==============================] - 4s 9ms/step - loss: 70832.7734 - mae: 70832.7734 - val_loss: 66870.3203 - val_mae: 66870.3203
Epoch 2/100
250/250 [==============================] - 1s 6ms/step - loss: 65740.2422 - mae: 65740.2422 - val_loss: 61252.8320 - val_mae: 61252.8320
Epoch 3/100
250/250 [==============================] - 2s 8ms/step - loss: 57757.1914 - mae: 57757.1914 - val_loss: 52954.3008 - val_mae: 52954.3008
Epoch 4/100
250/250 [==============================] - 2s 6ms/step - loss: 52296.7148 - mae: 52296.7148 - val_loss: 50600.5625 - val_mae: 50600.5625
Epoch 5/100
250/250 [==============================] - 2s 8ms/step - loss: 51395.5625 - mae: 51395.5625 - val_loss: 50966.7227 - val_mae: 50966.7227
Epoch 6/100
250/250 [==============================] - 1s 6ms/step - loss: 51152.1211 - mae: 51152.1211 - val_loss: 50460.1484 - val_mae: 50460.1484
Epoch 7/100
250/250 [==============================] - 1s 6ms/step - loss: 51184.0781 - mae: 51184.0781 

In [20]:
%load_ext tensorboard

In [23]:
%tensorboard --logdir regrassionlogs/fit/20260507-172205

In [22]:
# Evaluate the model on test data

test_loss, test_mae = model.evaluate(X_test,y_test)
print(f'Test_MAE:{test_mae }')

63/63 [==============================] - 0s 3ms/step - loss: 50313.7930 - mae: 50313.7930
Test_MAE:50313.79296875


In [ ]:
model.save("regression_model.h5")

d:\python\ANN\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
